In [2]:
import os
import json
from openai import OpenAI
from google.colab import userdata
from dotenv import load_dotenv
from openai import OpenAI

==============================================
## Initializing LLM Client
==============================================

In [3]:
load_dotenv()

try:
    client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))
except Exception as e:
    print(f"Error initializing ApenAI LLM client: {e} Please ensure that your environment variables are setup with a valid API key.")

==============================================
## Tool 1 - Domain Lookup Tool
==============================================

In [4]:
def lookup_domain_info(domain: str) -> str:

    print(f"-> Fetching domain info for {domain} from Domain database...")

    # Stubbed domain demo database
    mock_data = {
        "acmecorp.com": {"industry": "Software/SaaS", "size": "500-1000 employees", "revenue": "$50M - $100M"},
        "widgetco.net": {"industry": "Manufacturing", "size": "100-250 employees", "revenue": "$10M - $25M"},
        "globalfin.org": {"industry": "Financial Services", "size": "5000+ employees", "revenue": "$1B+"},
    }

    info = mock_data.get(domain, {"industry": "Unknown", "size": "N/A", "revenue": "N/A"})

    return json.dumps(info)

==============================================
## Tool 2 - Check CRM Hstory Tool
==============================================

In [5]:
def check_crm_history(email: str) -> str:

    print(f"-> Fetching CRM history for {email} from Lead database...")

    # Stubbed lead demo database
    mock_data = {
        "jane@acmecorp.com": {"last_contact": "2025-11-15", "status": "Cold Lead", "notes": "Attended webinar, no follow-up yet."},
        "bob@widgetco.net": {"last_contact": "2025-12-01", "status": "Active Opportunity", "notes": "Discussed Q1 budget and product integration."},
        "default": {"last_contact": "N/A", "status": "No Record", "notes": "New lead, first contact opportunity."},
    }

    history = mock_data.get(email, mock_data["default"])
    return json.dumps(history)

==============================================
## Tool 3 - Lead Score Calculator Tool
==============================================

In [6]:
def calculate_lead_score(data_summary: str) -> str:

    print("-> Fetching Calculated Lead Score...")

    data = json.loads(data_summary)
    score = "Low"

    # Stubbed demo scoring logic
    if data["domain_info"].get("revenue", "").startswith("$1B+"):
        score = "High"
    elif data["crm_history"].get("status") == "Active Opportunity":
        score = "High"
    elif data["domain_info"].get("revenue", "").startswith("$50M"):
        score = "Medium"

    return json.dumps({"lead_score": score})

==============================================
## Function-mapping
==============================================

In [7]:
AVAILABLE_FUNCTIONS = {
    "lookup_domain_info": lookup_domain_info,
    "check_crm_history": check_crm_history,
    "calculate_lead_score": calculate_lead_score,
}

==============================================
## Defining Tool Schemas
==============================================

In [8]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "lookup_domain_info",
            "description": "Retrieves general business information (industry, size, revenue) about a company based on its domain name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "domain": {"type": "string", "description": "The company's domain name, e.g., 'acmecorp.com'"},
                },
                "required": ["domain"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_crm_history",
            "description": "Checks the internal CRM system for past contact, status, and notes associated with a specific lead email.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {"type": "string", "description": "The full email address of the lead."},
                },
                "required": ["email"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_lead_score",
            "description": "Calculates the priority score (High/Medium/Low) for a lead based on a summary of all collected domain and CRM history data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "data_summary": {"type": "string", "description": "A JSON string containing the combined domain_info and crm_history."},
                },
                "required": ["data_summary"],
            },
        },
    },
]

=====================================================================
## Initialize Chain-of-Thought (CoT) Agentic Workflow
=====================================================================


In [9]:
def run_agent(user_prompt: str):
    """
    The main Chain-of-Thought (CoT) execution loop for the CRM Lead Qualifier Agent.
    """
    print(f"\n--- Running Lead Qualifier Agent ---")

    system_prompt = (
        "You are an expert CRM Lead Qualifier Agent. Your sole task is to analyze a sales lead "
        "provided via email address. You must follow these steps precisely: "
        "1. Identify the domain from the email. "
        "2. Call `lookup_domain_info` and `check_crm_history` sequentially to gather all data. "
        "3. Combine all collected data into a single JSON object. "
        "4. Call `calculate_lead_score` with the combined JSON object. "
        "5. Finally, synthesize all information (domain info, CRM history, and score) "
        "into a single, easy-to-read summary for a busy sales rep."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    collected_data = {}

    while True:
        print("\n[AI Thinking...]")
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto",
        )

        response_message = response.choices[0].message
        messages.append(response_message)

        if response_message.tool_calls:
            tool_calls = response_message.tool_calls

            for tool_call in tool_calls:
                function_name = tool_call.function.name
                function_to_call = AVAILABLE_FUNCTIONS.get(function_name)
                function_args = json.loads(tool_call.function.arguments)

                if not function_to_call:
                    print(f"Error: Unknown function {function_name}")
                    continue

                # Execute the function
                function_result = function_to_call(**function_args)

                # Update the persistent memory (collected_data)
                if function_name == "lookup_domain_info":
                    collected_data["domain_info"] = json.loads(function_result)
                elif function_name == "check_crm_history":
                    collected_data["crm_history"] = json.loads(function_result)
                elif function_name == "calculate_lead_score":
                    # Inject the accumulated data from previous turns
                    function_args = {"data_summary": json.dumps(collected_data)}
                    function_result = function_to_call(**function_args)

                # Append the tool result as a NEW message
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": function_result
                })

        else:
            print("\n--- FINAL AGENT SUMMARY ---")
            print(response_message.content)
            break

==============================================
## Execute Test Scenarios
==============================================

In [10]:
# Scenario 1: High-Value Lead (Large company, needs scoring)
lead_email_1 = "jane@acmecorp.com"
run_agent(f"Please qualify this lead for my call tomorrow: {lead_email_1}")

print("\n" + "="*80 + "\n")


--- Running Lead Qualifier Agent ---

[AI Thinking...]
-> Fetching domain info for acmecorp.com from Domain database...
-> Fetching CRM history for jane@acmecorp.com from Lead database...

[AI Thinking...]
-> Fetching Calculated Lead Score...
-> Fetching Calculated Lead Score...

[AI Thinking...]

--- FINAL AGENT SUMMARY ---
Here's a summary of the lead information for Jane at Acmecorp:

### Domain Information:
- **Industry:** Software/SaaS
- **Company Size:** 500-1000 employees
- **Revenue:** $50M - $100M

### CRM History:
- **Last Contact:** November 15, 2025
- **Status:** Cold Lead
- **Notes:** Jane attended a webinar but has not been followed up on yet.

### Lead Score:
- **Priority:** Medium

Jane appears to be part of a midsize software company with good revenue potential. Given that she is currently a cold lead who attended a webinar, a strategic follow-up could warm this lead. Consider personalizing your outreach to reignite interest.




In [11]:
# Scenario 2: Medium-Value Lead (Active opportunity, needs scoring)
lead_email_2 = "bob@widgetco.net"
run_agent(f"Can you run an analysis on this lead: {lead_email_2}")


--- Running Lead Qualifier Agent ---

[AI Thinking...]
-> Fetching domain info for widgetco.net from Domain database...
-> Fetching CRM history for bob@widgetco.net from Lead database...

[AI Thinking...]
-> Fetching Calculated Lead Score...
-> Fetching Calculated Lead Score...

[AI Thinking...]

--- FINAL AGENT SUMMARY ---
### Lead Analysis Summary

**Lead Information:**
- **Email:** bob@widgetco.net
- **Domain:** widgetco.net

**Domain Information:**
- **Industry:** Manufacturing
- **Company Size:** 100-250 employees
- **Revenue:** $10M - $25M

**CRM History:**
- **Last Contact Date:** December 1, 2025
- **Status:** Active Opportunity
- **Notes:** Discussed Q1 budget and product integration.

**Lead Score:** High

**Summary:**
Bob from WidgetCo.net represents a high-priority opportunity in the manufacturing sector. The company is of moderate size and revenue, with active discussions around budget and integration of products. Follow up quickly to capitalize on this promising lead.
